## RAG (Retrieval-Augmented Generation)
RAG는 기존 LLM을 확장하여 주어진 질문에 대해 더 정확하고 풍부한 정보를 제공하는 방법이다. 모델이 학습데이터에 포함되어 있지 않은 외부 데이터를 실시간으로 검색(retrieval)하고 이를 바탕으로 답변을 생성하는 과정을 포함한다.
RAG를 통해서
* 환각(Hallucination): 학습 데이터에 없는 내용을 그럴듯하게 지어내는 현상
* 최신 정보 부재: 학습 데이터 이후의 정보는 알 수 없음
* 도메인 지식 부족: 특정 회사/서비스의 내부 정보는 학습되지 않음
등의 문제를 해결할 수 있다.

In [1]:
!pip install -qqq langchain langchain-openai langchain_community tiktoken chromadb


지금 설치한 패키지는 5개 이다.
* langchain : LangChain의 핵심 패키지
* langchain-openai : OpenAI 모델(GPT 계열)을 LangChain에서 쉽게 쓰도록 해주는 패키지
* langchain_community : FAISS, Pinecone, Chroma 벡터스토어 연결
* tiktoken : OpenAI가 만든 토큰화 라이브러리로 텍스트를 토큰 단위로 나눌 때 사용
* chromadb : 임베딩을 저장하고 검색하는 데 쓰는 경량 벡터 데이터베이스

In [2]:
import langchain
langchain.__version__

'0.3.28'

In [3]:
# from google.colab import userdata
# import os

# os.environ["OPENAI_API_KEY"] = userdata.get("OPEN_AI")


# import os
# import getpass

# os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")


## RAG 파이프 라인 개요
* Load Data - Text Split - Indexing - Retrieval - Generation
* 데이터 읽어오고 - chunk단위로 text분할 - 벡터스토어로 indexing후 DB에 저장 -  검색 - 생성 순서이다

In [4]:
# Step1: 데이터 읽어오기
from langchain_community.document_loaders import WebBaseLoader  # HTML내용 가줘와 텍스트만 추출

# 위키피디아에서 유관순 검색
url = 'https://ko.wikipedia.org/wiki/%EC%9C%A0%EA%B4%80%EC%88%9C'
loader = WebBaseLoader(url)

docs = loader.load()
print(len(docs))
print("###########")
print(len(docs[0].page_content))
print("###########")
print(docs[0].page_content[2000:3000])


USER_AGENT environment variable not set, consider setting it to identify your requests.


1
###########
13046
###########
 형사들에게 붙잡혀 서대문형무소에서 이뤄진 모진 고문으로 인해 순국했다.
1916년 충청남도 공주시에서 선교활동을 하던 미국인 감리교회 선교사인 사애리시 부인(사부인)의 추천으로 이화학당 보통과 3학년에 장학생으로 편입하고, 1919년에 이화학당 고등부에 진학했다. 3월 1일 3.1 운동에 참여하고 3월 5일의 만세 시위에도 참여하였다. 총독부의 휴교령으로 천안으로 내려와 후속 만세 시위에 주도적으로 참여했다가 일제에 체포되어 공주지방법원에서 징역 5년형을 선고받고 항소하였고, 경성복심법원에서 징역 3년을 선고받아 사형이 확정되었다. 일제의 교도소에서 1920년 9월 28일에 순국했다.
1962년에 대한민국 건국훈장 독립장이 추서되었으며, 1996년에 이화여자고등학교는 명예 졸업장을 추서했다. 충청남도 천안시 동남구 병천면 용두리의 생가가 복원되어 1991년에 사적 제230호로 지정되었다. 천안 유관순 열사 유적과 천안종합운동장 내 '유관순체육관'[1]은 유관순의 이름을 딴 것이다. 해방 후 박인덕 등에 의해 기념사업이 추진되었는데, 이 때문에 박인덕 등이 자신들의 친일 의혹을 덮기 위한 불순한 의도로 이화학당 학생이었던 유관순 열사를 부각시켰다는 의혹도 제기되고 있다.
성명[편집]
본관은 고흥 유씨이다. 두음법칙과 관련하여 성명 표기에 대해 과거에 논란이 있었다. 2007년 4월에 일부 성씨의 사람들이 호적상 이름을 변경해 달라며 낸 신청을 받아들이는 지방법원의 결정이 있었고, 표기 문제에 대해 여러 국가기관에서 논의가 이루어졌다.[2] 논의의 결과 2007년 8월에 대법원의 호적예규[3]가 개정되었고 이에 따라 류씨의 성을 가진 사람은 본인 의사에 따라 한글로 '류'로 쓸 수 있게 되었다. 2007년 10월 현재는 개정된 호적예규가 문제시될 것이 없다고 확인을 했다.[4]
국가보훈처에 등록된 비영리 사단법인 유관순열사기념사업회는 처음에 ‘유’ 표기를 쓰고 있었고 2001년에 ‘류’로 바꾸었다. 그러나

In [5]:
!pip install langchain-text-splitters -q

In [6]:
# Step2: 데이터 청킹하기

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

splits = text_splitter.split_documents(docs)

print(len(splits))
print('-------------------------')
print(splits[10].page_content)
print('-------------------------')
print(splits[10].metadata)

19
-------------------------
1945년 광복 후 충청남도와 천안시의 협력으로 병천면에 유관순의 영정을 모신 사당이 건립되었다. 한편 1946년부터는 이화여고 교장이었던 신봉조와 박인덕 등 이화학당 출신 인사들에 의해 기념사업회가 조직되었다. 이 즈음, 서대문형무소로부터 유관순의 관을 인수한 이들이 상자를 열어보니 안에 든 시체가 토막으로 참살되었다는 소문 등이 퍼뜨려졌다.
1962년 3월 1일에 건국공로훈장 단장(후일 건국훈장 독립장으로 개정)이 추서되었다.
유관순의 유해가 망실된 관계로 1989년 10월 12일에 그녀의 고향인 충청남도 천안시 동남구 병천면 탑원리 매봉산 중턱에 가묘인 초혼묘(招魂墓)를 세웠다.
충청남도 천안시 병천면에 탄신 100주년을 맞아 유관순열사기념관(柳寬順烈士記念館)을 개관하여 유관순 열사의 영정을 봉안하고 2003년에 문을 열었다.
2019년 3월 1일에 건국훈장 대한민국장이 추서되었다.
연보[편집]
1902년 12월 16일 : 충청남도 천안 병천면에서 아버지 유중권(柳重權)과 어머니 이소제(李少悌)의 3남1녀 중 둘째로 태어났다.(이복언니가 있음)
아버지 유중권은 교육의 중요성을 깨닫고 흥호학교 설립에 참여했고, 작은아버지는 감리교회에서 일요일에만 교회에 올 수 있는 선교사들을 도와서 평일에 교회 일을 맡았다.
1916년 : 공주에서 전도하던 감리교 선교사 사 부인(미국이름 엘리스 샤프)의 권유로 이화학당 보통과에 입학하였다. 샤프 부인은 집안 사정이 어려운 유관순이 장학금을 받고 공부할 수 있도록 배려하였다.
1918년 : 보통과를 졸업하고, 이화학당 고등과에 입학하였다.
1919년 3월 1일 : 서울에서 3·1 운동에 참여했다. 학생들이 다칠 것을 걱정한 프라이 교장이 만류했으나, 학교 담을 넘어 3·1운동에 참여했다고 한다.
1919년 3월 8일 : 일제가 3월 10일부로 학교에 임시휴교령을 내리자, 같이 이화학당을 다니던 사촌언니 유예도와 함께 고향인 천안으로 내려갔다.
----------------

* chunk_size=1000
→ 텍스트를 최대 1000자 단위로 쪼갠다는 뜻
(너무 길면 모델 입력 한도를 초과할 수 있고, 검색 효율도 떨어지기 때문에 잘라줌)

* chunk_overlap=200
→ 잘린 조각들 사이에 200자 겹치는 부분을 유지
(문맥이 끊기지 않게 하기 위함. 예: 문단 중간에서 끊겼을 때 앞뒤 연결 보장)

* Recursive
→ 문단 → 문장 → 단어 순으로 가능한 "큰 단위부터" 쪼개고, 그래도 너무 길면 더 작은 단위로 쪼갬.
(예: 그냥 문자 단위로 무조건 잘라내면 문장이 끊어져 이해가 어렵지만, Recursive는 문맥 단위를 최대한 지켜줌)

In [7]:
! pip install langchain-upstage -q

 * Chroma : 문서를 임베딩(숫자 벡터로 변환)해서 저장하고,
나중에 "질문"이 들어오면 질문도 벡터로 바꿔서 가장 유사한 문서(Chunk) 를 찾아줌
 *  UpstageEmbeddings : 임베딩 모델을 불러오는 클래스 이며, 텍스트를 숫자 벡터로 변환해 준다.

In [8]:
# API 키를 설정합니다. Upstage API 키를 입력해주세요.
import os
import getpass

os.environ["UPSTAGE_API_KEY"] = getpass.getpass("여기에 UpStage API 키를 입력하시오: ")

여기에 UpStage API 키를 입력하시오:  ········


In [9]:
# Step3: 데이터 임베딩 하기

from langchain_community.vectorstores import Chroma
from langchain_upstage import UpstageEmbeddings

# 텍스트를 숫자 벡터로 변환해주는 '임베딩 모델'을 준비합니다.
embeddings = UpstageEmbeddings(model="embedding-query")

# 잘라낸 문서 조각(chunks)들을 임베딩하여 Vector DB에 저장합니다.
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings, persist_directory="./chroma_store")

docs = vectorstore.similarity_search("유관순 가족에 대해서 설명해주세요")
print(len(docs))
print("---------------------")
print(docs[1].page_content)
print("---------------------")
print(type(docs[1]))

4
---------------------
유관순 - 위키백과, 우리 모두의 백과사전































본문으로 이동







주 메뉴





주 메뉴
사이드바로 이동
숨기기



		둘러보기
	


대문최근 바뀜요즘 화제임의 문서로





		사용자 모임
	


사랑방사용자 모임관리 요청





		편집 안내
	


소개도움말정책과 지침질문방



















검색











검색






















보이기
















기부

계정 만들기

로그인








개인 도구





기부 계정 만들기 로그인




























목차
사이드바로 이동
숨기기




처음 위치





1
성명








2
업적




업적 하위섹션 토글하기





2.1
초기 활동








2.2
이화학당 재학과 만세 운동






2.2.1
이화학당








2.2.2
만세 운동










2.3
아우내 만세 운동 가담과 체포








2.4
투옥과 사인






2.4.1
재판








2.4.2
징역형 선고와 옥사








2.4.3
유관순 부모의 사인












3
사후








4
연보








5
논란과 의혹




논란과 의혹 하위섹션 토글하기





5.1
형량에 대한 중언부언








5.2
정치적, 종교적 목적의 악용 논란








5.3
이화학당 출신 인사의 은폐 의혹








5.4
유관순 우상화 배경 관련 의혹










6
가족 관계








7
유관순상/유관순 횃불상








8
유관순이 등장한 작품




유관순이 등장한 작품 하위섹션 토글하기





8.1
영화








8.2
애니메이션










9
기타








10
같이 보기








11
관련 자료








12
각주






출력이 너무 지저분 하죠?

지금 우리 한것이 뭐에요?
1. UpstageEmbeddings로 문장을 벡터로 변환
2. Chroma로 그 벡터들을 데이터베이스로 저장해 빠르게 검색할 수 있게 만들었고
3. similarity_search() 써서 주어진 질문 문장과 의미적으로 가장 비슷한 문서들을 반환했어요.

그런데 결과에 공백이랑 잡다한 텍스트가 너무 많네요.
HTML 문서 원문을 그대로 임베딩 하면 이렇게 됩니다.
전처리가 조금 필요해 보이죠?









```
def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)
```
이러한 전처리를 하는 함수 하나 만들 것입니다.

지금 docs는 `Document`라는 파이썬 객체에요. 그리고 그 안에 page_content (본문) 하고 metadata (출처정보) 가 들어있는 형태 에요.
```
Context=[Document(page_content='유관순은 3.1운동에 참여한 독립운동가이다.'), Document(page_content='그녀는 만세운동을...')]
```

LLM은 이런 객체 구조를 바로 이해를 못해요. 그래서 문맥을 이해 못해서 이상하게 답변을 출력한 겁니다.

함수는 간단해요. Document안에 공백을 딱 두칸만 엔터를 누른것 처럼 해서 하나로 합치는 거지요.


자 이제 검색하고 답변 생성해 볼까요?

먼저 LLM 모델을 GPT로 써 볼꼐요. API키 입력 부터 해보죠


In [10]:
# from google.colab import userdata
# import os

# os.environ["OPENAI_API_KEY"] = userdata.get("OPEN_AI")


import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")


OpenAI API Key:  ········


In [11]:
# Step4: 검색하고 생성하기

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


# 프롬프트
template = '''
아래 context를 기반으로 응답을 해주세요:
{context}

Question: {question}
'''

prompt = ChatPromptTemplate.from_template(template)

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# .as_retriever() : 벡터DB를 검색용 객체로 변환
retriever=vectorstore.as_retriever()

# 전처리 함수를 통해 각 문서 사이를 줄바꿈으로 구분해 prompt용 문자열로 만들어
def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

# RAG
# Runnable 적용
# retriever | format_docs 전처리 과정은 여러개의 retriever 문서를 하나의 문서로 전처리한다.

# 어! 러너블람다 안썼는데?
# 호출가능한 객체는 러너블람다를 안써도 자동으로 감싼다. 그래서 그냥 함수명 써도 오케이 입니다.

chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

result=chain.invoke("유관순 가족에 대해서 설명해주세요")
print(result)
print("**********************************************************************")
# LLM 통해서 출력한 결과 (그냥 비교한번 해보기)
result=model.invoke("유관순 가족에 대해서 설명해주세요")
result.content

유관순의 가족은 다음과 같습니다:

- **부**: 유중권
- **모**: 이소제
- **형**: 유우석
- **동생**: 유관석

유관순은 1902년 12월 16일에 태어나, 독립운동가로 활동하였으며, 그녀의 가족은 그녀의 독립운동에 큰 영향을 미쳤습니다. 유관순의 부모님은 그녀의 독립운동에 대한 지지를 보였고, 그녀의 형제들도 그녀와 함께 독립운동에 참여한 것으로 알려져 있습니다.
**********************************************************************


'유관순(1902-1947)은 한국의 독립운동가로, 1919년 3.1 운동에 참여하여 큰 영향을 미친 인물입니다. 그녀는 충청남도 천안에서 태어나, 일본의 식민지 지배에 저항하기 위해 독립운동에 헌신했습니다. 유관순은 특히 3.1 운동 당시 천안에서 주도적인 역할을 하였고, 그로 인해 일본 경찰에 체포되어 고문을 받기도 했습니다.\n\n유관순의 가족에 대한 정보는 상대적으로 적지만, 그녀의 아버지 유중권은 독립운동에 대한 의지를 가지고 있었던 인물로 알려져 있습니다. 유관순은 가족의 영향을 받아 독립운동에 참여하게 되었고, 그녀의 가족은 그녀의 활동을 지지했습니다. 유관순은 독립운동을 위해 많은 희생을 감수했으며, 그녀의 이름은 한국의 독립운동 역사에서 중요한 상징으로 남아 있습니다.\n\n유관순은 1920년대 초반에 일본 경찰에 의해 체포되어 감옥에서 고문을 당한 후, 1920년 9월 28일에 사망했습니다. 그녀의 용기와 희생은 오늘날에도 많은 사람들에게 영감을 주고 있으며, 그녀의 생애는 한국의 독립운동 역사에서 중요한 위치를 차지하고 있습니다.'

RAG를 통해서 위키피디아에 있는 정확한 정보를 가져올 수 있었습니다.
그냥 LLM에 물어보는 것 보다는 더 나은 답변이지요??

## PDF 파일을 가상컴퓨터에 업로드 하자.


In [15]:
#!pip install -q pypdf # 코랩버전
!pip install -q pypdf langchain-community # 주피터랩 버전

In [13]:
# 코랩에서의 파일 업로드 (주피터랩에서는 생략)

# import os
# from google.colab import files

# uploaded = files.upload()  # 파일 선택창이 뜸
# print(os.getcwd())   # 현재 작업 디렉토리가 어딘지 출력하는 코드 "/content"
# print(os.listdir())  # 현재 작업중인 디렉토리에 파일 목록 확인


## PDF 파일 데이터 가져오기

langchain_community 패키지에서 제공하는 PyPDFLoader를 사용하여 PDF 파일에서 텍스트를 추출합니다. 이 명령을 사용하려면 pypdf 라이브러리를 먼저 설치해야 한다.

In [14]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma #벡터db
from langchain_upstage import UpstageEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


# Step1: 데이터 읽어오기 (data_loading)

pdf_filepath = 'SK_2023.pdf'
loader = PyPDFLoader(pdf_filepath)
docs = loader.load()

# len(docs)
# docs[10]

# Step2: text분할 ( Documents 를 청킹하기)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs) # 청크단위로 조각내

# Step3: 임베딩하기 (Indexing 하기)
embeddings = UpstageEmbeddings(model="embedding-query")
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings, persist_directory="./chroma_store")
docs = vectorstore.similarity_search("sk 회장 누구니?")

# print(len(docs))
# print(docs[1].page_content)

# Step4: 검색하고 생성하기

# Prompt
template = '''
아래 context를 기반으로 응답을 해주세요. MBTI가 완전 T처럼 응답해줘:
{context}

Question: {question}
'''

prompt = ChatPromptTemplate.from_template(template)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# vectorstore : 문서들을 벡터로 변환해 저장해둔 벡터DB
# .as_retriever() : 벡터DB를 검색용 객체로 변환
retriever=vectorstore.as_retriever()

# 전처리 함수를 통해 각 문서 사이를 줄바꿈으로 구분해 prompt용 문자열로 만들어 주기
# 만약에 그냥 retriever를 바로 프롬프트에 넣으면 LLM이 Document객체를 처리하지 못해서 결과가 이상하게 나오기도함
def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

# RAG
# LangChain의 Runnable 스타일 문법이 적용됨
# retriever | format_docs 전처리 과정은 여러개의 retriever 문서를 하나의 문서로 전처리한다.
chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

result=chain.invoke("sk그룹 23년 지속가능 경영을 위한 노력?")
print(result)

print("**********************************************************************")

# RAG를 이용하지 않은 LLM 통해서 출력한 결과 (비교한번 해보기)
result=model.invoke("sk그룹 23년 지속가능 경영을 위한 노력?")
result.content


SK그룹은 2023년 지속가능 경영을 위해 다음과 같은 주요 노력을 기울이고 있습니다:

1. **탄소 감축 목표 설정**: SK그룹은 2030년까지 전 세계 탄소배출량의 1%인 약 2억 톤의 CO2를 감축하겠다는 목표를 수립했습니다. 이는 파리기후 협약의 1.5도 합의 이행을 위한 중요한 이정표입니다.

2. **친환경 사업 확대**: SK주식회사는 친환경 비즈니스에 대한 직접 투자 비중을 30%로 확대하며, 기후변화 대응 및 그린전환을 위한 다양한 친환경 사업을 추진하고 있습니다. 주요 투자 영역으로는 ICT 에너지 솔루션, 친환경 모빌리티, 재사용 포장재, 전기차 공급망 관리, 배터리 소재, 수소 및 소형모듈형 원전 등이 포함됩니다.

3. **내부 탄소 가격제 도입**: SK는 경영활동 전반에 탄소 사용에 대한 내부 가격을 설정하여 저탄소 기술 투자를 촉진하고 있습니다. 이를 통해 탄소 리스크를 관리하고, 지속 가능한 성장 모델을 구축하고자 합니다.

4. **RE100 이행**: SK그룹은 재생에너지 사용을 100%로 전환하기 위해 다양한 프로젝트를 추진하고 있으며, 해외 태양광 비즈니스 구축 및 재생에너지 사업 확대를 통해 탄소 배출권 등 부가 수익을 창출하고 있습니다.

5. **글로벌 파트너십 강화**: SK는 글로벌 공급망 재편에 대응하기 위해 주요 대륙 및 국가에서 친환경 사업을 추진하고, 글로벌 파트너사와 협업하여 경쟁력을 강화하고 있습니다. 특히, 미국과 유럽 중심으로 성장 잠재력이 높은 기술을 보유한 기업에 대한 직접 투자 및 공급망 파트너링을 확대하고 있습니다.

이러한 노력들은 SK그룹이 지속 가능한 저탄소 경제로의 전환을 가속화하고, 기후위기 해결에 기여하기 위한 전략적 접근을 반영하고 있습니다.
**********************************************************************


'SK그룹은 2023년에도 지속가능 경영을 위한 다양한 노력을 기울이고 있습니다. 주요 내용은 다음과 같습니다:\n\n1. **탄소 중립 목표**: SK그룹은 2050년까지 탄소 중립을 달성하기 위한 로드맵을 수립하고, 이를 위한 구체적인 실행 계획을 마련하고 있습니다. 이를 통해 온실가스 배출을 줄이고, 재생 가능 에너지 사용을 확대하고자 합니다.\n\n2. **ESG 경영 강화**: 환경(Environment), 사회(Social), 지배구조(Governance) 측면에서의 지속 가능성을 강화하기 위해 ESG 경영을 적극적으로 추진하고 있습니다. 이를 통해 기업의 투명성과 책임성을 높이고, 사회적 가치를 창출하고자 합니다.\n\n3. **친환경 기술 개발**: SK그룹은 친환경 기술 및 제품 개발에 투자하고 있으며, 이를 통해 지속 가능한 산업 생태계를 구축하고 있습니다. 예를 들어, 전기차 배터리, 수소 에너지 등 미래 지향적인 기술에 집중하고 있습니다.\n\n4. **사회적 책임**: SK그룹은 지역 사회와의 상생을 위해 다양한 사회 공헌 활동을 진행하고 있습니다. 교육, 복지, 환경 보호 등 다양한 분야에서 사회적 가치를 창출하기 위한 프로그램을 운영하고 있습니다.\n\n5. **지속 가능한 공급망 관리**: 공급망의 지속 가능성을 높이기 위해 협력업체와의 협력을 강화하고, 윤리적이고 지속 가능한 조달 방식을 채택하고 있습니다.\n\n이와 같은 노력들은 SK그룹이 지속 가능한 미래를 위해 나아가는 방향성을 보여주며, 기업의 사회적 책임을 다하기 위한 지속적인 노력을 반영하고 있습니다.'